<a href="https://colab.research.google.com/github/yuukienomoto/report_ynu/blob/main/mahjong_0521.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 20人のID取得
雀魂牌譜屋の開発者ツールから「ranking」と検索→「4w」をクリックしてリンクを取得。

In [ ]:
import requests
import pandas as pd

def fetch_top_player_ids():
    # 正解のURL
    RANKING_URL = "https://5-data.amae-koromo.com/api/v2/pl4/player_delta_ranking/4w"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    print("ランキングデータの取得を開始します...")

    try:
        response = requests.get(RANKING_URL, headers=headers)
        response.raise_for_status()

        ranking_data = response.json()
        strong_player_ids = []

        # ★修正ポイント：階層をたどって、実際のプレイヤーリストを取り出す
        # "0" というキーの中の、"top" というキーにリストが入っている
        player_list = ranking_data.get("0", {}).get("top", [])

        # 上位100人分だけループしてIDを抽出
        for player in player_list[:100]:
            player_id = player.get('id')
            player_name = player.get('nickname')

            if player_id:
                strong_player_ids.append(player_id)
                print(f"ID取得: {player_id} ({player_name})")

        print(f"\n合計 {len(strong_player_ids)} 人の強者IDを取得しました！")

        # 文字化け防止のために encoding='utf-8-sig' を追加してCSV保存
        df = pd.DataFrame({'player_id': strong_player_ids})
        df.to_csv("strong_players_list.csv", index=False, encoding='utf-8-sig')
        print("リストを 'strong_players_list.csv' に保存しました。")

        return strong_player_ids

    except Exception as e:
        print(f"エラーが発生しました: {e}")
        return []

if __name__ == "__main__":
    ids = fetch_top_player_ids()

ランキングデータの取得を開始します...
ID取得: 22894632 (狂日日记)
ID取得: 69951433 (inorimachi)
ID取得: 16210656 (ぜっふちょう)
ID取得: 22972779 (KairoxBoy)
ID取得: 107678699 (123azx)
ID取得: 15631679 (招小伟)
ID取得: 359092 (魂天おっと)
ID取得: 15382278 (東方鏡月)
ID取得: 19962975 (SUB02)
ID取得: 23729152 (榮斷藥)
ID取得: 13713752 (9374号路人)
ID取得: 11080050 (叫陰天別鬧了)
ID取得: 19895545 (因果律幻神)
ID取得: 9649449 (花花哥哥丶)
ID取得: 103300048 (pug01)
ID取得: 104926315 (@和華)
ID取得: 123937219 (Darkney)
ID取得: 22385265 (别想捉我炮GU)
ID取得: 110619 (韶华未央)
ID取得: 14774809 (槿の花一日です)

合計 20 人の強者IDを取得しました！
リストを 'strong_players_list.csv' に保存しました。


# 取得したIDを使って詳細成績を集める
## 一人一人の和了率や放銃率などのパラメータを自動で取得する

In [ ]:
import requests
import pandas as pd
import time
import random

def fetch_detailed_stats():
    # 1. 先ほど保存したCSVからIDを読み込む
    try:
        df_ids = pd.read_csv("strong_players_list.csv")
        player_ids = df_ids['player_id'].tolist()
        print(f"CSVから {len(player_ids)} 人のIDを読み込みました。")
    except FileNotFoundError:
        print("CSVファイルが見つかりません。先にランキング取得コードを実行してください。")
        return

    # 2. 個別成績を取得するためのURLテンプレート
    # 取得していただいた完璧なURLを設定します
    PLAYER_URL_TEMPLATE = "https://5-data.amae-koromo.com/api/v2/pl4/player_stats/16210656/1262304000000/1779347339999?mode=16.12.9.15.11.8&tag=494263"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    dataset = []
    print("\n個別成績の取得を開始します...（※サーバー配慮のため少し時間がかかります）")

    for pid in player_ids:
        try:
            # プレイヤーごとのURLを作成
            # テンプレート内の "16210656" を、現在ループしているID（pid）に書き換える
            url = PLAYER_URL_TEMPLATE.replace("16210656", str(pid))

            response = requests.get(url, headers=headers)
            response.raise_for_status()
            data = response.json()

            # 3. 機械学習（LightGBM）に使う特徴量を抽出
            features = {
                'player_id': pid,
                'is_strong': 1,  # 今回はランキング上位の強者リストなので1
                'game_count': data.get('count', 0),
                'win_rate': data.get('hora_rate', 0.0),      # 和了率
                'deal_in_rate': data.get('houju_rate', 0.0), # 放銃率
                'call_rate': data.get('furo_rate', 0.0),     # 副露率
                'riichi_rate': data.get('riichi_rate', 0.0), # リーチ率
            }
            dataset.append(features)
            print(f"取得成功: ID {pid}")

            # 4. 【超重要】サーバー負荷軽減のための待機時間（2〜4秒）
            time.sleep(random.uniform(2.0, 4.0))

        except Exception as e:
            print(f"エラー発生 ID {pid}: {e}")
            time.sleep(5) # エラー時も少し待つ

    # 5. データ保存
    df_stats = pd.DataFrame(dataset)
    df_stats.to_csv("strong_players_stats.csv", index=False, encoding='utf-8-sig')

    print("\n--- 収集完了 ---")
    print("詳細データを 'strong_players_stats.csv' に保存しました。")

    # 最初の5行だけプレビュー表示
    print("\n【取得データプレビュー】")
    print(df_stats.head())

if __name__ == "__main__":
    fetch_detailed_stats()

CSVから 20 人のIDを読み込みました。

個別成績の取得を開始します...（※サーバー配慮のため少し時間がかかります）
取得成功: ID 22894632
取得成功: ID 69951433
取得成功: ID 16210656
取得成功: ID 22972779
取得成功: ID 107678699
取得成功: ID 15631679
取得成功: ID 359092
取得成功: ID 15382278
取得成功: ID 19962975
取得成功: ID 23729152
取得成功: ID 13713752
取得成功: ID 11080050
取得成功: ID 19895545
取得成功: ID 9649449
取得成功: ID 103300048
取得成功: ID 104926315
取得成功: ID 123937219
取得成功: ID 22385265
取得成功: ID 110619
取得成功: ID 14774809

--- 収集完了 ---
詳細データを 'strong_players_stats.csv' に保存しました。

【取得データプレビュー】
   player_id  is_strong  game_count  win_rate  deal_in_rate  call_rate  \
0   22894632          1         329       0.0           0.0        0.0   
1   69951433          1        1314       0.0           0.0        0.0   
2   16210656          1        1073       0.0           0.0        0.0   
3   22972779          1          86       0.0           0.0        0.0   
4  107678699          1         367       0.0           0.0        0.0   

   riichi_rate  
0          0.0  
1          0.0  
2          

In [ ]:
import requests
import json

# あなたが取得してくれた正しいURL
url = "https://5-data.amae-koromo.com/api/v2/pl4/player_stats/16210656/1262304000000/1779347339999?mode=16.12.9.15.11.8&tag=494263"
# ID部分を1人目の「22894632」に変えて中身をチェック
url = url.replace("16210656", "22894632")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
data = response.json()

print("--- データのトップレベルにある名前一覧 ---")
print(data.keys())

print("\n--- データの中身のサンプル（最初の300文字） ---")
print(json.dumps(data, ensure_ascii=False)[:300])

--- データのトップレベルにある名前一覧 ---
dict_keys(['count', 'level', 'max_level', 'rank_rates', 'rank_avg_score', 'avg_rank', 'negative_rate', 'id', 'nickname', 'played_modes'])

--- データの中身のサンプル（最初の300文字） ---
{"count": 329, "level": {"id": 10502, "score": 5033, "delta": 144}, "max_level": {"id": 10502, "score": 5033, "delta": 144}, "rank_rates": [0.3556231003039514, 0.270516717325228, 0.24620060790273557, 0.1276595744680851], "rank_avg_score": [43896, 30582, 20743, 6724], "avg_rank": 2.1458966565349544, 
